# Model 2: Improved MLP with Hyperparameter Tuning

Improvement over Assignment 1 baseline MLP (~77% accuracy).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import re
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# ── Load Data ──────────────────────────────────────────────────────────────
df = pd.read_csv('IMDB Dataset.csv')
df['label'] = (df['sentiment'] == 'positive').astype(int)
print(df.shape)
df.head()

In [ ]:
# ── Feature Engineering (extended) ────────────────────────────────────────
def clean_text(text):
    text = re.sub(r'<.*?>', ' ', text)          # remove HTML tags
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)    # keep letters only
    return text.lower().strip()

analyzer = SentimentIntensityAnalyzer()

def extract_features(text):
    clean = clean_text(text)
    vader  = analyzer.polarity_scores(text)
    blob   = TextBlob(clean)
    return {
        'vader_compound':  vader['compound'],
        'vader_pos':       vader['pos'],
        'vader_neg':       vader['neg'],
        'vader_neu':       vader['neu'],
        'textblob_polar':  blob.sentiment.polarity,
        'textblob_subj':   blob.sentiment.subjectivity,
        'review_length':   len(clean.split()),
        'exclaim_count':   text.count('!'),
        'question_count':  text.count('?'),
    }

print('Extracting features...')
features = df['review'].apply(extract_features)
X = pd.DataFrame(list(features)).values.astype(np.float32)
y = df['label'].values.astype(np.float32)

# Normalise
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val   = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
# ── DataLoaders ────────────────────────────────────────────────────────────
def make_loader(X, y, batch_size=256, shuffle=True):
    ds = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train)
val_loader   = make_loader(X_val,   y_val,   shuffle=False)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)

In [ ]:
# ── Improved MLP Architecture ──────────────────────────────────────────────
class ImprovedMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)

model = ImprovedMLP(input_dim=9, hidden_dims=[64, 32, 16], dropout=0.3)
print(model)

In [ ]:
# ── Training with Adam + LR Scheduler ──────────────────────────────────────
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss   = criterion(logits, yb)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(yb)
            preds       = (torch.sigmoid(logits) >= 0.5).long()
            correct    += (preds == yb.long()).sum().item()
            n          += len(yb)
    return total_loss / n, correct / n

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val, best_state, patience_cnt, PATIENCE = 0, None, 0, 15

for epoch in range(1, 101):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader,   train=False)
    scheduler.step(vl)
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    if va > best_val:
        best_val   = va
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | Train Loss {tl:.4f} Acc {ta:.4f} | Val Loss {vl:.4f} Acc {va:.4f}')

model.load_state_dict(best_state)

In [ ]:
# ── Evaluation ─────────────────────────────────────────────────────────────
_, test_acc = run_epoch(test_loader, train=False)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(device))
        preds  = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
        all_preds.extend(preds); all_labels.extend(yb.long().numpy())

print(f'Test Accuracy: {test_acc:.4f}')
print(classification_report(all_labels, all_preds, target_names=['Negative','Positive']))

In [ ]:
# ── Plots ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss Curves'); axes[0].legend()

axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'],   label='Val Acc')
axes[1].set_title('Accuracy Curves'); axes[1].legend()

plt.tight_layout()
plt.savefig('../results/improved_mlp_curves.png', dpi=150)
plt.show()

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'], yticklabels=['Negative','Positive'])
plt.title('Confusion Matrix – Improved MLP')
plt.tight_layout()
plt.savefig('../results/improved_mlp_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── Save Results ───────────────────────────────────────────────────────────
report = classification_report(all_labels, all_preds, target_names=['Negative','Positive'])
with open('../results/improved_mlp_results.txt', 'w') as f:
    f.write('=== Improved MLP Results ===\n\n')
    f.write(f'Test Accuracy: {test_acc:.4f}\n\n')
    f.write('Classification Report:\n')
    f.write(report)
    f.write(f'\nBest Val Accuracy: {best_val:.4f}\n')
print('Results saved.')